Trabajo Final de Depósito y Mineria de Datos - Cursada 2024  
 Integrantes:  
    Francisco Repetto FAI-2548   
    Malena Rivera     FAI-2516

In [1]:
#importamos las librerias necesarias
import pandas as pd
import numpy as np

# Limpieza de CSV (Datos Estructurados)

#### Leemos el csv sucio que deseamos limpiar

In [2]:
df_credito = pd.read_csv('credit_card_fraud_10k_dirty.csv')

In [3]:
#vemos las caracteristicas del csv leido
df_credito.info()
df_credito.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 10300 entries, 0 to 10299
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   transaction_id       9813 non-null   str  
 1   amount               9812 non-null   str  
 2   transaction_hour     9803 non-null   str  
 3   merchant_category    9796 non-null   str  
 4   foreign_transaction  9797 non-null   str  
 5   location_mismatch    9815 non-null   str  
 6   device_trust_score   9786 non-null   str  
 7   velocity_last_24h    9788 non-null   str  
 8   cardholder_age       9808 non-null   str  
 9   is_fraud             9800 non-null   str  
dtypes: str(10)
memory usage: 804.8 KB


,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
count,9813,9812,9803,9796,9797,9815,9786,9788,9808,9800
unique,9228,8185,29,11,4,4,77,12,54,4
top,error_transaction_id,error_amount,14.0,Food,0.0,0.0,45.0,1.0,29.0,0.0
freq,198,201,428,2003,8738,8869,155,2614,212,9546


Como podemos observar, el csv cuenta con 10300 filas y cuenta con 10 columnas las cuales son:  
1. transaccion_id: identificador unico  
2. cantidad: monto de la transaccion  
3. hora_de_transaccion: hora de la transaccion (0-23)  
4. categoría_comercial: tipo de comerciante  
5. transaccion_extranjera: Indica si la transaccion es internacional (0/1)  
6. discrepancia_de_ubicacion: desajuste entre facturacion y ubicacion de transaccion (0/1)
7. confianza_del_dispositivo_usado: puntuacion de confianza del dispositivo (0-100)
8. velocidad_ultimas_24h: Numero de transacciones en las ultimas 24hs
9. edad_del_titular: edad del titular de la tarjeta
10. es_fraude: Variable objetivo. Indica si es fraude o no (0=normal | 1=fraude)

Por otro lado, tambien podemos observar que muchas de las filas deben ser numericas tal como se describe anteriormente. Sin embargo, todas las columnas son del tipo str, es decir, que se guardo texto y no numeros. 

Procederemos a la limpieza del mismo

### ETAPA 1 - COERCION DE TIPOS  

Vamos a cambiar el tipo de todas aquellas columnas que deben ser numericas y actualmente son del tipo str.
En el caso de que haya algun error durante la convercion se dejará en blanco esa celda (errors=coerce)

In [4]:
#etapa 1- coercion (forzamos todo a numero)
cols_numericas = [
    'transaction_id', 'amount', 'transaction_hour', 
    'foreign_transaction', 'location_mismatch', 
    'device_trust_score', 'velocity_last_24h', 
    'cardholder_age', 'is_fraud'
]
for col in cols_numericas:
    df_credito[col] = pd.to_numeric(df_credito[col], errors='coerce')

In [5]:
#comprobamos el tipo ahora
df_credito.info()

<class 'pandas.DataFrame'>
RangeIndex: 10300 entries, 0 to 10299
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       9586 non-null   float64
 1   amount               9597 non-null   float64
 2   transaction_hour     9587 non-null   float64
 3   merchant_category    9796 non-null   str    
 4   foreign_transaction  9796 non-null   float64
 5   location_mismatch    9814 non-null   float64
 6   device_trust_score   9782 non-null   float64
 7   velocity_last_24h    9787 non-null   float64
 8   cardholder_age       9802 non-null   float64
 9   is_fraud             9799 non-null   float64
dtypes: float64(9), str(1)
memory usage: 804.8 KB


Podemos ver ahora que las columnas numericas son del tipo float64.  
Eso puede NO ser del todo correcto ya que hay algunas columnas que no deben tener decimales como por ejemplo 'cardholder_age' que guarda la edad del titular o 'is_fraud' que es una variable booleana y solo guarda valores 0/1

In [6]:
#veamos ahora la cantidad de nulos por columna
print("valores nulos por columna:")
print (df_credito.isnull().sum())

valores nulos por columna:
transaction_id         714
amount                 703
transaction_hour       713
merchant_category      504
foreign_transaction    504
location_mismatch      486
device_trust_score     518
velocity_last_24h      513
cardholder_age         498
is_fraud               501
dtype: int64


Empezemos por la primer columna

In [7]:
print("antes")
df_credito["transaction_id"].unique()[:10]

antes


array([1.00000000e+00, 2.00000000e+00, 3.00000000e+00, 2.49876498e+05,
       5.00000000e+00, 6.00000000e+00, 7.00000000e+00, 8.00000000e+00,
       9.00000000e+00,            nan])

El id deberia ser incremental y del tipo entero, por lo tanto, vamos a cambiar todas las filas que sean raras para que sigan el orden preestablecido

In [8]:
#ordenamos por id
df_credito = df_credito.sort_values("transaction_id").reset_index(drop=True)

#eliminamos valores invalidos (por ejemplo, valores con decimales)
df_credito= df_credito[df_credito["transaction_id"] %1 ==0]

#ahora reconstruimos la secuencia
df_credito["transaction_id"] = range(1, len(df_credito) +1)


In [9]:
print("despues")
df_credito["transaction_id"].unique()[:10]

despues


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

### ETAPA 2 - Tratamiento de Nulos

In [10]:
#vemos cuantos valores faltantes tiene la columna 'is_fraud'
print(df_credito['is_fraud'].value_counts(dropna=False))

is_fraud
0.000000    8788
NaN          465
1.000000     138
0.765619      92
Name: count, dtype: int64


Podemos observar que no solo hay valores nulos (Nan) sino valores extraños  
Recordemos que esta columna solo debe guardar valores 0 y 1 donde 0 indica que NO hubo fraude y 1 que sí

Vamos a llenar con 0 (dado que es el valor mas comun) a las filas vacias  
Aquellas que tienen un valor extraño las borraremos dado que son pocas

In [11]:
# Vamos a llenar las celdas en blanco con 0, es decir, que no hubo fraude
df_credito['is_fraud'] = df_credito['is_fraud'].fillna(0)

antes_nulos = len(df_credito)

# Nos quedamos SOLO con las filas donde el valor es exactamente 0 o 1
# Esto elimina cualquier "valor extraño" (como 2, 99, o strings raros)
df_credito = df_credito[df_credito['is_fraud'].isin([0, 1])]

df_credito['is_fraud'] = df_credito['is_fraud'].astype(int)

despues = len(df_credito)
print(f"Filas eliminadas en is_fraud: {antes_nulos - despues}")

print(df_credito['is_fraud'].value_counts(dropna=False))

Filas eliminadas en is_fraud: 92
is_fraud
0    9253
1     138
Name: count, dtype: int64


Como podemos ver hay más cantidad de 0 que de 1, eso indica que va a estar más influenciado de que no hay fraude de que sí

A continuacion, aplicaremos la misma logica con el resto de variable booleanas

In [12]:
#Empezamos con 'foreing_transaction' que indica si la transaccion fue extranjera (1) o no (0)
#Veamos la cantidad de nulos y si tiene valores raros
print(df_credito['foreign_transaction'].value_counts(dropna=False))

foreign_transaction
0.000000    7958
1.000000     877
NaN          463
4.923391      93
Name: count, dtype: int64


In [13]:
#solo debe tener 1 y 0 asi que arreglamos eso
df_credito['foreign_transaction'] = df_credito['foreign_transaction'].fillna(0) #sacamos los nulos

# 1. Los nulos los podemos borrar directamente (o podés dejarlos si querés que pasen al filtro de abajo)
antes_nulos = len(df_credito)

# Nos quedamos SOLO con las filas donde el valor es exactamente 0 o 1
# Esto elimina cualquier "valor extraño" (como 2, 99, o strings raros)
df_credito = df_credito[df_credito['foreign_transaction'].isin([0, 1])]

df_credito['foreign_transaction'] = df_credito['foreign_transaction'].astype(int)

despues =  len(df_credito)
print(f"Filas eliminadas en foreing_transaction: {antes_nulos - despues}")


Filas eliminadas en foreing_transaction: 93


In [14]:
#despues
print(df_credito['foreign_transaction'].value_counts(dropna=False))

foreign_transaction
0    8421
1     877
Name: count, dtype: int64


In [15]:
#Nuevamente la misma logica ahora con location_mismatch que indica si hay discrepancias entre la facturacion y la compra
#Veamos la cantidad de Nulos y si tiene valores extraños
print(df_credito['location_mismatch'].value_counts(dropna=False))

location_mismatch
0.00000    8007
1.00000     747
NaN         443
4.30973     101
Name: count, dtype: int64


In [16]:
#hacemos lo mismo que con las otras dos
#solo debe tener 1 y 0 asi que arreglamos eso
df_credito['location_mismatch'] = df_credito['location_mismatch'].fillna(0) #sacamos los nulos

# 1. Los nulos los podemos borrar directamente (o podés dejarlos si querés que pasen al filtro de abajo)
antes_nulos = len(df_credito)

# Nos quedamos SOLO con las filas donde el valor es exactamente 0 o 1
# Esto elimina cualquier "valor extraño" (como 2, 99, o strings raros)
df_credito = df_credito[df_credito['location_mismatch'].isin([0, 1])]

df_credito['location_mismatch'] = df_credito['location_mismatch'].astype(int)

despues = len(df_credito)
print(f"Filas eliminadas en location_mismatch: {antes_nulos - despues}")

Filas eliminadas en location_mismatch: 101


In [17]:
#despues
print(df_credito['location_mismatch'].value_counts(dropna=False))

location_mismatch
0    8450
1     747
Name: count, dtype: int64


Para las variables continuas tambien debemos tratar sus nulos y valores extraños  
En estos casos rellenaremos los nulos con la mediana

In [18]:
#Primero veamos cuantos nulos tiene cada columna
cols_rellenar_mediana = ['amount', 'cardholder_age', 'device_trust_score', 'velocity_last_24h']
for col in cols_rellenar_mediana:
   print(col,"tiene:", df_credito[col].isnull().sum(),"nulos")

amount tiene: 632 nulos
cardholder_age tiene: 450 nulos
device_trust_score tiene: 469 nulos
velocity_last_24h tiene: 456 nulos


In [19]:
#Rellenemos estos valores nulos con la mediana como dijimos antes
for col in cols_rellenar_mediana:
   df_credito[col] = df_credito[col].fillna(df_credito[col].median())

#y verificamos ahora la cantidad de nulos
   print(col,"tiene:", df_credito[col].isnull().sum(),"nulos")

amount tiene: 0 nulos
cardholder_age tiene: 0 nulos
device_trust_score tiene: 0 nulos
velocity_last_24h tiene: 0 nulos


In [20]:
#Sin embargo, tambien debemos verificar si algunas de estas columnas tiene valores atipicos

for col in cols_rellenar_mediana:
    # Calculamos el Primer Cuartil (25%) y el Tercer Cuartil (75%)
    Q1 = df_credito[col].quantile(0.25)
    Q3 = df_credito[col].quantile(0.75)
    # Calculamos el rango intercuartilico (IQR) es la diferencia entre Q3 y Q1 (el ancho del 50% de los datos centrales)
    IQR = Q3 - Q1

    #Definimos los umbrales estadísticos para detectar valores "extremos"
    # El estándar es 1.5 veces el IQR por arriba y por debajo
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
   # Identificamos los registros que están fuera de esos límites (Outliers)
    outliers = df_credito[(df_credito[col] < limite_inferior) | (df_credito[col] > limite_superior)]

    #Imprimimos el reporte de resultados para la columna actual
    print(f"Columna: {col}")
    print(f"Cantidad de outliers detectados: {len(outliers)}")
    print(f"Límite superior: {limite_superior}")

    # Mostramos los primeros 10 valores que se pasaron de los límites
    print(outliers[col].head(10))
    
    # Comparamos con la mediana y el máximo para ver qué tan dispersos están los datos
    print(f"Mediana: {df_credito[col].median()}")
    print(f"Valor máximo: {df_credito[col].max()}") 
    
    print("-" * 30)

Columna: amount
Cantidad de outliers detectados: 589
Límite superior: 504.2199999999999
1      541.82
9      630.64
31     535.26
32     535.26
50     628.71
52     504.92
69     780.15
127    515.36
132    642.20
133    590.76
Name: amount, dtype: float64
Mediana: 123.88
Valor máximo: 8813.402647488014
------------------------------
Columna: cardholder_age
Cantidad de outliers detectados: 89
Límite superior: 93.5
65      2168.551759
140     2168.551759
196     2168.551759
382     2168.551759
388     2168.551759
618     2168.551759
990     2168.551759
1035    2168.551759
1085    2168.551759
1328    2168.551759
Name: cardholder_age, dtype: float64
Mediana: 44.0
Valor máximo: 2168.5517593064765
------------------------------
Columna: device_trust_score
Cantidad de outliers detectados: 97
Límite superior: 132.5
447     3091.598528
579     3091.598528
689     3091.598528
778     3091.598528
875     3091.598528
1082    3091.598528
1253    3091.598528
1254    3091.598528
1289    3091.598528


In [21]:
# Definimos las columnas que están "rotas" (edad, score y velocidad)
# Hacemos una copia por seguridad antes de borrar
df_limpio = df_credito.copy()

cols_a_limpiar = ['cardholder_age', 'device_trust_score', 'velocity_last_24h', 'amount']

for col in cols_a_limpiar:
    #Calculamos los cuartiles y el rango intercuartílico (IQR) para la columna actual
    Q1 = df_limpio[col].quantile(0.25)
    Q3 = df_limpio[col].quantile(0.75)
    IQR = Q3 - Q1
    
   # Ajustamos la sensibilidad de la limpieza:
    # Para la columna amount usamos un factor de 3.0  para no borrar tanto.
    # Para el resto, usamos el estándar de 1.5
    factor = 3.0 if col == 'amount' else 1.5

    #Calculamos los límites de corte
    limite_inferior = Q1 - factor * IQR
    limite_superior = Q3 + factor * IQR
    
    antes = len(df_limpio)

    # Aplicamos una máscara booleana para conservar solo lo que está entre los límites
    # Usamos '&' para que se cumplan AMBAS condiciones (mayor al inferior Y menor al superior).
    df_limpio = df_limpio[(df_limpio[col] >= limite_inferior) & (df_limpio[col] <= limite_superior)]
    
    despues = len(df_limpio)
    
    print(f"Columna {col}: Se eliminaron {antes - despues} filas.")

print(f"\nTotal de filas eliminadas: {len(df_credito) - len(df_limpio)}")
print(f"Tamaño actual del dataset: {len(df_limpio)} filas.")

Columna cardholder_age: Se eliminaron 89 filas.
Columna device_trust_score: Se eliminaron 95 filas.
Columna velocity_last_24h: Se eliminaron 135 filas.
Columna amount: Se eliminaron 186 filas.

Total de filas eliminadas: 505
Tamaño actual del dataset: 8692 filas.


In [22]:
#Como se trabajo sobre una copia ahora hacemos que nuestro dataset sea la copia
df_credito = df_limpio

In [23]:
#Para ver como quedo volvemos a ver los outliers de las columnas trabajadas anteriormente

for col in cols_rellenar_mediana:
    Q1 = df_credito[col].quantile(0.25)
    Q3 = df_credito[col].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    # Filtramos los outliers
    outliers = df_credito[(df_credito[col] < limite_inferior) | (df_credito[col] > limite_superior)]
    
    print(f"Columna: {col}")
    print(f"Cantidad de outliers detectados: {len(outliers)}")
    print(f"Límite superior: {limite_superior}")
    print(outliers[col].head()) # Ver los primeros 5 valores locos
    print(f"Mediana: {df_credito[col].median()}")
    print(f"Valor máximo: {df_credito[col].max()}")
    print("-" * 30)

Columna: amount
Cantidad de outliers detectados: 479
Límite superior: 479.04875000000004
1      541.82
9      630.64
50     628.71
52     504.92
127    515.36
Name: amount, dtype: float64
Mediana: 123.88
Valor máximo: 773.35
------------------------------
Columna: cardholder_age
Cantidad de outliers detectados: 0
Límite superior: 93.5
Series([], Name: cardholder_age, dtype: float64)
Mediana: 44.0
Valor máximo: 69.0
------------------------------
Columna: device_trust_score
Cantidad de outliers detectados: 0
Límite superior: 131.5
Series([], Name: device_trust_score, dtype: float64)
Mediana: 62.0
Valor máximo: 99.0
------------------------------
Columna: velocity_last_24h
Cantidad de outliers detectados: 0
Límite superior: 6.0
Series([], Name: velocity_last_24h, dtype: float64)
Mediana: 2.0
Valor máximo: 6.0
------------------------------


Como podemos notar, antes de la limpieza de valores tipicos habia bastantes inconsistencias. Por ejemplo
1. Habia transaccion con cantidades de 8813.402647488014
2. Habia titulares de la tarjeta con edades de 2168.5517593064765 años
3. Habia valores en la columna de las horas del dia (0-23) con valores de 100.6646216768916

Todo eso quedo erradicado con la limpieza hecha anteriormente. Podemos notar que ahora, por ejemplo, la cantidad más alta de una transaccion es de 773.35, la edad del titular mas grande es de 69 años pero se considera atipico si supera los 93.5 años

In [24]:
#Vamos ahora con la columna de categoria de la compra. Esta columna tiene todos sus valores con texto
#Veamos si tiene nulos o valores extraños

print(df_credito['merchant_category'].value_counts(dropna=False))

merchant_category
Food           1706
Clothing       1659
Travel         1601
Electronics    1564
Grocery        1562
NaN             427
FOOD             38
ELECTRONICS      38
CLOTHING         30
GROCERY          30
TRAVEL           27
NAN              10
Name: count, dtype: int64


Podemos ver que hay filas que tienen categoria nan y otras NAN
- nan es el nulo de python es decir que no tienen nada
- NAN sin embargo, es texto, auqnue se quiere dar a entender q son valores vacios, por lo tanto lo remplazaremos

Tambien hay dos valores para la misma categoria, nada mas que uno está en mayusculas y otras en minusculas. Tambien limpiaremos eso

In [25]:
# Convertimos el string 'NAN' a nulo real
df_credito['merchant_category'] = df_credito['merchant_category'].replace('NAN', np.nan)

# Pasamos todo a minúsculas y capitalizamos
# El .str automáticamente ignora los nulos reales (NaN)
df_credito['merchant_category'] = df_credito['merchant_category'].str.title()

print(df_credito['merchant_category'].value_counts(dropna=False))

merchant_category
Food           1744
Clothing       1689
Travel         1628
Electronics    1602
Grocery        1592
NaN             437
Name: count, dtype: int64


In [26]:
#Ahora trateremos los nulos llenando esos valores con la Moda (el valor más tipico)
#Como podemos observar de la celda anterior, el valor más tipico es Comida (food), asi que todas estas celdas ahora serán de Comida

moda_mercancia = df_credito['merchant_category'].mode()[0]
df_credito['merchant_category'] = df_credito['merchant_category'].fillna(moda_mercancia)
print(df_credito['merchant_category'].value_counts(dropna=False))

merchant_category
Food           2181
Clothing       1689
Travel         1628
Electronics    1602
Grocery        1592
Name: count, dtype: int64


In [27]:
#vamos ahora con la ultima columna: HORA de la transaccion
#Como siempre, veamos que valores tiene
print(df_credito['transaction_hour'].value_counts(dropna=False))

transaction_hour
NaN           593
14.000000     365
7.000000      357
23.000000     351
13.000000     349
18.000000     347
1.000000      347
9.000000      347
8.000000      347
15.000000     345
22.000000     341
3.000000      340
5.000000      339
17.000000     338
0.000000      337
21.000000     333
19.000000     333
16.000000     332
20.000000     332
12.000000     328
4.000000      315
6.000000      313
2.000000      304
11.000000     300
10.000000     283
578.740609     76
Name: count, dtype: int64


In [28]:
#hay 593 nulos asi q vamos a reemplazar eso por la moda
moda_hora = df_credito['transaction_hour'].mode()[0]
df_credito['transaction_hour'] = df_credito['transaction_hour'].fillna(moda_hora)
print(df_credito['transaction_hour'].value_counts(dropna=False))

transaction_hour
14.000000     958
7.000000      357
23.000000     351
13.000000     349
18.000000     347
1.000000      347
9.000000      347
8.000000      347
15.000000     345
22.000000     341
3.000000      340
5.000000      339
17.000000     338
0.000000      337
21.000000     333
19.000000     333
16.000000     332
20.000000     332
12.000000     328
4.000000      315
6.000000      313
2.000000      304
11.000000     300
10.000000     283
578.740609     76
Name: count, dtype: int64


In [29]:
#podemos ver que hay un valor atipico de hora 578, lo vamos a borrar
# Filtramos para quedarnos solo con lo que NO sea 578.740609 
# Filtramos SIN comillas porque es un número
# Usamos un margen pequeño por si hay problemas de redondeo en floats
# Borramos cualquier valor que sea mayor a 24 (porque una hora no puede ser 578!)
antes = len(df_credito)
df_credito = df_credito[df_credito['transaction_hour'] <= 24] 
despues = len(df_credito)

print(f"Se eliminaron {antes - despues} filas.")

df_credito['transaction_hour'] = df_credito['transaction_hour'].astype(int)
print(df_credito['transaction_hour'].value_counts())



Se eliminaron 76 filas.
transaction_hour
14    958
7     357
23    351
13    349
18    347
1     347
9     347
8     347
15    345
22    341
3     340
5     339
17    338
0     337
21    333
19    333
16    332
20    332
12    328
4     315
6     313
2     304
11    300
10    283
Name: count, dtype: int64


### ETAPA 3: CONVERSIÓN DE TIPOS (Formato Final)

In [30]:
# Ahora que no hay nulos ni basura, convertimos a enteros limpios, algunos ya están pero por las dudas
cols_a_entero = [
    'transaction_id', 'transaction_hour', 'cardholder_age', 
    'is_fraud', 'foreign_transaction', 'location_mismatch'
]
for col in cols_a_entero:
    df_credito[col] = df_credito[col].astype(int)

In [31]:
df_credito.info()

<class 'pandas.DataFrame'>
Index: 8616 entries, 0 to 9482
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       8616 non-null   int64  
 1   amount               8616 non-null   float64
 2   transaction_hour     8616 non-null   int64  
 3   merchant_category    8616 non-null   str    
 4   foreign_transaction  8616 non-null   int64  
 5   location_mismatch    8616 non-null   int64  
 6   device_trust_score   8616 non-null   float64
 7   velocity_last_24h    8616 non-null   float64
 8   cardholder_age       8616 non-null   int64  
 9   is_fraud             8616 non-null   int64  
dtypes: float64(3), int64(6), str(1)
memory usage: 740.4 KB


In [32]:
#por las dudas eliminamos duplicados exactos
antes = len(df_credito)
df_credito.drop_duplicates(inplace=True)
despues = len(df_credito)
print(f"Se eliminaron {antes - despues} filas.")

Se eliminaron 0 filas.


In [33]:
print("valores nulos por columna:")
print (df_credito.isnull().sum())

valores nulos por columna:
transaction_id         0
amount                 0
transaction_hour       0
merchant_category      0
foreign_transaction    0
location_mismatch      0
device_trust_score     0
velocity_last_24h      0
cardholder_age         0
is_fraud               0
dtype: int64


In [34]:
# Guardamos el DataFrame limpio en un archivo nuevo
df_credito.to_csv('datos_credito_limpios.csv', index=False)
df_credito.info()


<class 'pandas.DataFrame'>
Index: 8616 entries, 0 to 9482
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       8616 non-null   int64  
 1   amount               8616 non-null   float64
 2   transaction_hour     8616 non-null   int64  
 3   merchant_category    8616 non-null   str    
 4   foreign_transaction  8616 non-null   int64  
 5   location_mismatch    8616 non-null   int64  
 6   device_trust_score   8616 non-null   float64
 7   velocity_last_24h    8616 non-null   float64
 8   cardholder_age       8616 non-null   int64  
 9   is_fraud             8616 non-null   int64  
dtypes: float64(3), int64(6), str(1)
memory usage: 740.4 KB


### Etapa 4 - Subirlo a Clickhouse
Una vez listo el csv procederemos a subirlo a ClickHouse

Para este proyecto estamos siguiendo un flujo de ETL (Extract, Transform, Load). Actualmente estamos en la fase de Transformación, preparando el terreno para la Carga final en el Data Warehouse.

¿Qué es ClickHouse y por qué lo usamos?
Es una base de datos orientada a columnas optimizada para analítica (OLAP). A diferencia de una base transaccional, ClickHouse procesa millones de filas por segundo, lo que lo hace ideal como motor de nuestro Warehouse.

La importancia de la limpieza y estructura:

1. Rigidez del Warehouse: Al ser un sistema estructurado, ClickHouse exige que cada dato coincida exactamente con su tipo (no podés mandar un texto donde va un número).

2. Eficiencia: El motor vuela porque comprime los datos. Si los datos están sucios o mal formateados, esa eficiencia se pierde.

3. Calidad Analítica: Limpiar outliers y normalizar categorías (como unificar mayúsculas) es lo que garantiza que los reportes y tableros finales sean confiables. Sin una buena estructura previa, el Warehouse no sirve.

In [35]:
!pip install clickhouse_connect # en el caso de no tener instalado clickhouse
import clickhouse_connect

In [36]:
# nos conectamos al servidor
#Recordar tener levantado el servidor desde docker
client = clickhouse_connect.get_client(host='host.docker.internal', username='defaul', password='admin', port=8123)

In [37]:
#creamos la tabla
client.command('DROP TABLE IF EXISTS mi_tabla_limpia')

# 2. Creamos la tabla con el esquema completo basado en tus columnas
client.command('''
CREATE TABLE mi_tabla_limpia (
    transaction_id Int64,
    amount Float64,
    transaction_hour Int8,
    merchant_category String,
    foreign_transaction Int8,
    location_mismatch Int8,
    device_trust_score Float64,
    velocity_last_24h Float64,
    cardholder_age Int16,
    is_fraud Int8
) ENGINE = MergeTree() 
ORDER BY transaction_id
''')

In [38]:
#Insertar el DataFrame directamente
client.insert_df('mi_tabla_limpia', df_credito)

# Limpieza de Jsons (Datos semi-estructurados)

In [39]:
import time
import pandas as pd

t0= time.time()

### Carga inicial de los archivos json

In [40]:
#leemos todos los archivos jsons

import glob
import os

#Obtenemos la lista de todos los archivos JSON en la carpeta
path=path = "./jsons/*.json"
files = glob.glob(path)

# Leemos y concatenamos
# El parámetro lines=True depende de si el JSONs es un objeto por archivo o JSON-Lines.
list_df = []
for f in files:
    try:
        temp_df = pd.read_json(f) # Si es un JSON por archivo
        list_df.append(temp_df)
    except Exception as e:
        # Aquí simulamos el "columnNameOfCorruptRecord" de Spark
        print(f"Error en archivo {f}: {e}")

df_raw = pd.concat(list_df, ignore_index=True)

#Equivalente a df_raw.printSchema()
print("--- Estructura del DataFrame (Schema) ---")
print(df_raw.info())

#Equivalente a df_raw.limit(5).show()
print("\n--- Primeras 5 filas ---")
print(df_raw.head(5))


--- Estructura del DataFrame (Schema) ---
<class 'pandas.DataFrame'>
RangeIndex: 5006 entries, 0 to 5005
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   transaction_id     5005 non-null   object 
 1   timestamp          5005 non-null   object 
 2   transaction_hour   4951 non-null   object 
 3   amount             4952 non-null   object 
 4   merchant_category  5006 non-null   str    
 5   cardholder         5005 non-null   object 
 6   device             5006 non-null   object 
 7   network_features   5006 non-null   object 
 8   location           5006 non-null   object 
 9   authentication     5006 non-null   object 
 10  labels             5006 non-null   object 
 11  schema_version     92 non-null     float64
dtypes: float64(1), object(10), str(1)
memory usage: 469.4+ KB
None

--- Primeras 5 filas ---
     transaction_id                  timestamp transaction_hour  \
0               NaN          

Como podemos observar, los JSON cuentan con 5005 filas y 12 columnas, las cuales son:
- transaction_id: identificador único de la transacción.
- timestamp: fecha y hora de la transacción.
- transaction_hour: hora de la transacción (0–23).
- amount: monto de la transacción.
- merchant_category: categoría del comercio.
- cardholder: información del titular de la tarjeta (edad y país).
- device: información del dispositivo usado (tipo, sistema operativo, score de confianza).
- network_features: características de la red (velocidad de transacciones, riesgo de IP).
- location: ciudad y país de la transacción, indica si es extranjera.
- authentication: método de autenticación y cantidad de intentos fallidos.
- labels: variable objetivo indicando si es fraude (0=normal | 1=fraude).
- schema_version: versión del esquema del dataset (muchos valores nulos).

Por otro lado, podemos notar que varias columnas que deberían ser numéricas (transaction_hour, amount, device_trust_score, velocity_last_24h, ip_risk_score) se encuentran como texto u objetos

Antes de comenzar con el proceso de limpieza, se realiza una inspección inicial del dataset con el objetivo de comprender su estructura, verificar el formato de las columnas y detectar posibles inconsistencias visibles a simple vista

In [41]:
# Spark: df_raw.select(df_raw.columns[:5]).show(10, truncate=False)
print(df_raw[df_raw.columns[:5]].head(10))

     transaction_id                  timestamp transaction_hour  \
0               NaN                        NaT              NaN   
1  TX-2024-00077201  2024-09-23 22:58:07+00:00               22   
2  TX-2024-00098321  2024-05-12 14:32:10+00:00               14   
3  TX-2024-00000001  2024-02-06 09:18:10+00:00               13   
4  TX-2024-00000002  2024-01-03 12:34:22+00:00             15.0   
5  TX-2024-00000003  2024-09-30 15:05:07+00:00              578   
6  TX-2024-00000004  2024-04-14 20:05:45+00:00               15   
7  TX-2024-00000005  2024-12-20 15:02:43+00:00               13   
8  TX-2024-00000006  2024-06-15 09:17:33+00:00               16   
9  TX-2024-00000007  2024-09-21 06:32:40+00:00               14   

               amount merchant_category  
0  999999999999.98999            travel  
1                 420            travel  
2               245.5       electronics  
3              168.04          clothing  
4               84.09         groceries  
5         

In [42]:
#Listamos columnas
df_raw.columns

Index(['transaction_id', 'timestamp', 'transaction_hour', 'amount',
       'merchant_category', 'cardholder', 'device', 'network_features',
       'location', 'authentication', 'labels', 'schema_version'],
      dtype='str')

In [43]:
#y ahora listamos cantidad de filas
df_raw.count()

transaction_id       5005
timestamp            5005
transaction_hour     4951
amount               4952
merchant_category    5006
cardholder           5005
device               5006
network_features     5006
location             5006
authentication       5006
labels               5006
schema_version         92
dtype: int64

Podemos notar que algunas columnas presentan menos o más registros que el total de filas (5005), lo que indica la presencia de valores faltantes o inconsistentes.

### Analizamos si se cargaron bien todos los archivos

In [44]:
# Spark: df_raw.where("transaction_id = 'TX-2024-00098321'").select(...)

# El archivo transaccion1.json tenia la transaccion con id=TX-2024-00098321
filtro = df_raw['transaction_id'] == 'TX-2024-00098321'
columnas = ["transaction_id", "timestamp", "transaction_hour", "amount"]

# Aplicamos filtro y seleccionamos columnas al mismo tiempo
df_resultado = df_raw.loc[filtro, columnas]

print(df_resultado)

        transaction_id                  timestamp transaction_hour amount
2     TX-2024-00098321  2024-05-12 14:32:10+00:00               14  245.5
5003  TX-2024-00098321  2024-05-12 14:32:10+00:00               14  245.5


Se observa que la transacción se encuentra presente en el dataframe, lo que confirma que el archivo fue correctamente cargado.
Sin embargo, se detecta que el mismo transaction_id aparece en dos filas distintas, lo que podría indicar:

In [45]:
#Archivo transaccion2.json tenia la transaccion con id=TX-2024-00011123
columnas_interes = ["transaction_id", "timestamp", "transaction_hour", "amount"]

resultado = df_raw[df_raw["transaction_id"] == "TX-2024-00011123"][columnas_interes]

print(resultado)

        transaction_id                  timestamp transaction_hour amount
5004  TX-2024-00011123  2024-03-02 10:15:41+00:00               10   54.3


In [46]:
##Archivo transaccion3.json tenia la transaccion con id=TX-2024-00045877
df_resultado = df_raw.loc[
    df_raw["transaction_id"] == "TX-2024-00045877", 
    ["transaction_id", "timestamp", "transaction_hour", "amount"]
]

print(df_resultado)

        transaction_id                  timestamp transaction_hour   amount
5005  TX-2024-00045877  2024-07-11 03:42:19+00:00                3  1299.99


In [47]:
#Archivo transaccion4.json tenia la transaccion con id=TX-2024-00077201
cols = ["transaction_id", "timestamp", "transaction_hour", "amount"]

# Filtramos la transacción 4
df_tx4 = df_raw.loc[df_raw["transaction_id"] == "TX-2024-00077201", cols]

print(df_tx4)

     transaction_id                  timestamp transaction_hour amount
1  TX-2024-00077201  2024-09-23 22:58:07+00:00               22    420


In [48]:
#Probamos archivo transaccion6.json tenia una transaccion con id=null pero un amount reconocible
filtro_monto = df_raw["amount"] == 999999999999.99
columnas = ["transaction_id", "timestamp", "transaction_hour", "amount"]

df_anomala = df_raw.loc[filtro_monto, columnas]

# Mostramos el resultado
print(df_anomala)

  transaction_id timestamp transaction_hour              amount
0            NaN       NaT              NaN  999999999999.98999


Hemos corroborado que todos los archivos se han leido correctamente, asi que sigamos la limpieza

#### Sacamos las estructuras internas y aplanamos (flatten). Tambien casteamos

En esta etapa se construye un nuevo dataframe (df1) a partir del dataset original (df_raw), con el objetivo de:
- Eliminar filas completamente vacías.
- Desanidar columnas que contienen diccionarios.
- Convertir variables al tipo de dato correspondiente.

In [49]:
# -------------------------------------------------------------------
# 1. Eliminación de filas completamente vacías
# Spark ignora automáticamente estas filas, Pandas no.
# Para mantener coherencia entre entornos, las eliminamos manualmente.
# -------------------------------------------------------------------

df_raw = df_raw.dropna(how='all').copy()


# -------------------------------------------------------------------
# 2. Creación de nuevo DataFrame estructurado (df1)
# Aquí desanidamos los diccionarios y convertimos tipos de datos.
# -------------------------------------------------------------------

df1 = pd.DataFrame()

# ------------------- TRANSACTION -------------------

df1['transaction_id'] = df_raw['transaction_id']

# Convertimos timestamp a formato datetime
df1['timestamp'] = pd.to_datetime(df_raw['timestamp'], errors='coerce')

# Convertimos a numérico las variables cuantitativas
df1['amount'] = pd.to_numeric(df_raw['amount'], errors='coerce')
df1['transaction_hour'] = pd.to_numeric(df_raw['transaction_hour'], errors='coerce')


# ------------------- MERCHANT -------------------

df1['merchant_category'] = df_raw['merchant_category']


# ------------------- CARDHOLDER -------------------

df1['cardholder_age'] = pd.to_numeric(
    df_raw['cardholder'].apply(lambda x: x.get('age') if isinstance(x, dict) else None),
    errors='coerce'
)

df1['cardholder_country'] = df_raw['cardholder'].apply(
    lambda x: x.get('country') if isinstance(x, dict) else None
)


# ------------------- DEVICE -------------------

df1['device_type'] = df_raw['device'].apply(
    lambda x: x.get('device_type') if isinstance(x, dict) else None
)

df1['device_os'] = df_raw['device'].apply(
    lambda x: x.get('operating_system') if isinstance(x, dict) else None
)

df1['device_trust_score'] = pd.to_numeric(
    df_raw['device'].apply(lambda x: x.get('device_trust_score') if isinstance(x, dict) else None),
    errors='coerce'
)


# ------------------- NETWORK -------------------

df1['velocity_last_24h'] = pd.to_numeric(
    df_raw['network_features'].apply(lambda x: x.get('velocity_last_24h') if isinstance(x, dict) else None),
    errors='coerce'
)

df1['ip_risk_score'] = pd.to_numeric(
    df_raw['network_features'].apply(lambda x: x.get('ip_risk_score') if isinstance(x, dict) else None),
    errors='coerce'
)


# ------------------- LOCATION -------------------

df1['city'] = df_raw['location'].apply(
    lambda x: x.get('city') if isinstance(x, dict) else None
)

df1['country'] = df_raw['location'].apply(
    lambda x: x.get('country') if isinstance(x, dict) else None
)

# Normalizamos variable booleana similar a lógica Spark
def spark_logic(val):
    if val is None:
        return None
    v = str(val).strip().lower()
    if v in ["true", "1", "1.0", "yes"]:
        return True
    if v in ["false", "0", "0.0", "no"]:
        return False
    return None

df1['is_foreign_transaction'] = (
    df_raw['location']
    .apply(lambda x: x.get('is_foreign_transaction') if isinstance(x, dict) else None)
    .apply(spark_logic)
)

df1['location_mismatch'] = (
    df_raw['location']
    .apply(lambda x: x.get('location_mismatch') if isinstance(x, dict) else None)
    .map({'true': True, 'false': False, True: True, False: False})
)


# ------------------- AUTHENTICATION -------------------

df1['auth_method'] = df_raw['authentication'].apply(
    lambda x: x.get('method') if isinstance(x, dict) else None
)

df1['failed_attempts'] = pd.to_numeric(
    df_raw['authentication'].apply(lambda x: x.get('failed_attempts') if isinstance(x, dict) else None),
    errors='coerce'
)


# ------------------- LABEL Y VERSION -------------------

df1['is_fraud'] = df_raw['labels'].apply(
    lambda x: x.get('is_fraud') if isinstance(x, dict) else None
)

df1['schema_version'] = df_raw['schema_version']


# Verificación de nulos
print("Cantidad de valores nulos por columna:")
print(df1.isna().sum())

# Verificación de tipos de datos
print("\nTipos de datos actuales:")
print(df1.dtypes)


Cantidad de valores nulos por columna:
transaction_id               1
timestamp                    1
amount                      84
transaction_hour            82
merchant_category            0
cardholder_age              61
cardholder_country           1
device_type                  0
device_os                    0
device_trust_score          57
velocity_last_24h           73
ip_risk_score                0
city                         0
country                      0
is_foreign_transaction      54
location_mismatch          119
auth_method                  0
failed_attempts              0
is_fraud                     0
schema_version            4914
dtype: int64

Tipos de datos actuales:
transaction_id                         object
timestamp                 datetime64[us, UTC]
amount                                float64
transaction_hour                      float64
merchant_category                         str
cardholder_age                        float64
cardholder_country        

Podemos observar ahora si que ya no todas las variables son del tipo string sino que de su tipo correspondiente. Por ejemplo, amount, transaction_hour, device_trust_score, velocity_last_24h, failed_attempt ahora son variables numericas

Algunas columnas presentan valores faltantes, especialmente amount, transaction_hour, cardholder_age y schema_version.

In [50]:
df1.columns

Index(['transaction_id', 'timestamp', 'amount', 'transaction_hour',
       'merchant_category', 'cardholder_age', 'cardholder_country',
       'device_type', 'device_os', 'device_trust_score', 'velocity_last_24h',
       'ip_risk_score', 'city', 'country', 'is_foreign_transaction',
       'location_mismatch', 'auth_method', 'failed_attempts', 'is_fraud',
       'schema_version'],
      dtype='str')

### Analizamos la cantidad de nulos, columnas sobrantes, duplicados

In [51]:
# Spark: Todo ese bloque con stack y selectExpr...
# Pandas:

# 1. Calculamos los nulos por columna
null_counts = df1.isna().sum()

# 2. Lo convertimos en un DataFrame para que se vea igual al "stack" de Spark
df_nulls = null_counts.reset_index()
df_nulls.columns = ['column', 'null_count']

# Mostramos el resultado
print(df_nulls)

                    column  null_count
0           transaction_id           1
1                timestamp           1
2                   amount          84
3         transaction_hour          82
4        merchant_category           0
5           cardholder_age          61
6       cardholder_country           1
7              device_type           0
8                device_os           0
9       device_trust_score          57
10       velocity_last_24h          73
11           ip_risk_score           0
12                    city           0
13                 country           0
14  is_foreign_transaction          54
15       location_mismatch         119
16             auth_method           0
17         failed_attempts           0
18                is_fraud           0
19          schema_version        4914


In [52]:
# Borramos schema_version y creamos un nuevo dataframe que sea df2
df2 = df1.drop(columns=['schema_version'])

# Mostramos las columnas para confirmar
print(df2.columns)

Index(['transaction_id', 'timestamp', 'amount', 'transaction_hour',
       'merchant_category', 'cardholder_age', 'cardholder_country',
       'device_type', 'device_os', 'device_trust_score', 'velocity_last_24h',
       'ip_risk_score', 'city', 'country', 'is_foreign_transaction',
       'location_mismatch', 'auth_method', 'failed_attempts', 'is_fraud'],
      dtype='str')


Del print anterior tambien podemos ver que hay un unico nulo en la columna de transaction_id asi que procederemos a  borrar esa fila

In [53]:
# 1. Traducción literal de: df2.na.drop(subset=["transaction_id"])
df2 = df2.dropna(subset=["transaction_id"])

# 2. Traducción literal del conteo y el "stack" de Spark
# En Spark hacés un select con una lista de sumas, acá hacemos el sum()
null_counts_series = df2.isnull().sum()

# Convertimos a DataFrame (el equivalente al selectExpr con stack)
df_nulls = null_counts_series.reset_index()
df_nulls.columns = ['column', 'null_count']

# Mostramos el resultado (el .show() de Spark)
print(df_nulls.to_string(index=False))

                column  null_count
        transaction_id           0
             timestamp           0
                amount          84
      transaction_hour          81
     merchant_category           0
        cardholder_age          60
    cardholder_country           0
           device_type           0
             device_os           0
    device_trust_score          57
     velocity_last_24h          73
         ip_risk_score           0
                  city           0
               country           0
is_foreign_transaction          54
     location_mismatch         118
           auth_method           0
       failed_attempts           0
              is_fraud           0


In [54]:
#La columna amount tampoco tiene que tener nulos. Borramos esas filas
# Spark: df2.na.drop(subset=["amount"])
df2 = df2.dropna(subset=["amount"])

# Traducción literal del bloque de conteo y stack
null_counts_series = df2.isnull().sum()

# Replicamos el selectExpr(stack...) para que quede como tabla
df_nulls = null_counts_series.reset_index()
df_nulls.columns = ['column', 'null_count']

# Mostramos el resultado (el .show() de Spark)
print(df_nulls.to_string(index=False))

                column  null_count
        transaction_id           0
             timestamp           0
                amount           0
      transaction_hour          76
     merchant_category           0
        cardholder_age          58
    cardholder_country           0
           device_type           0
             device_os           0
    device_trust_score          51
     velocity_last_24h          63
         ip_risk_score           0
                  city           0
               country           0
is_foreign_transaction          50
     location_mismatch         103
           auth_method           0
       failed_attempts           0
              is_fraud           0


In [55]:
#Verificamos si hay filas duplicadas por id de transaccion
aux = df2.groupby("transaction_id").size().reset_index(name='count')
aux = aux[aux['count'] > 1]

# Spark: aux.show()
print(aux)

# Spark: print(aux.count())
# En Pandas, count() cuenta elementos por columna, para el total de filas usamos len() o .shape[0]
print(len(aux))

        transaction_id  count
14    TX-2024-00000016      2
22    TX-2024-00000024      2
30    TX-2024-00000034      2
32    TX-2024-00000036      2
36    TX-2024-00000041      2
...                ...    ...
3553  TX-2024-00003708      2
3677  TX-2024-00003836      2
3948  TX-2024-00004120      2
4106  TX-2024-00004288      2
4796  TX-2024-00098321      2

[121 rows x 2 columns]
121


In [56]:
#Eliminamos duplicados por id de transaccion
df2 = df2.drop_duplicates(subset=["transaction_id"])

# Verificamos si hay filas duplicadas (Traducción literal)
# Spark: aux = df2.groupBy("transaction_id").count().filter("count > 1")
aux = df2.groupby("transaction_id").size().reset_index(name='count')
aux = aux[aux['count'] > 1]

# Cantidad de ids duplicados
# Spark: print(aux.count())
print(len(aux))

0


### Validamos rangos de los distintas columnas

En esta etapa se analizan los valores mínimos y máximos de las columnas numéricas del dataset. El objetivo es:
- Detectar valores fuera de rango (por ejemplo, horas mayores a 23).
- Identificar montos negativos o inconsistentes.
- Verificar que las variables cuantitativas respeten los límites esperados.

In [57]:
#Calculamos los rangos de valores de cada columna
# 1. Identificamos columnas numéricas (Equivalente a num_cols de Spark)
num_cols = df2.select_dtypes(include=['number']).columns

# 2. Calculamos min y max, transponemos (.T) y reseteamos el índice
# Esto hace exactamente lo mismo que el agg + stack de Spark
df_rangos = df2[num_cols].agg(['min', 'max']).T.reset_index()

# 3. Renombramos las columnas para que queden igual que en tu alias de Spark
df_rangos.columns = ['column', 'min', 'max']

# 4. Mostramos el resultado (Equivalente a .show)
print(df_rangos.to_string(index=False))

            column   min         max
            amount  0.01 8813.402647
  transaction_hour  0.00  578.740609
    cardholder_age 18.00 2168.551759
device_trust_score 25.00 3091.598528
 velocity_last_24h  0.00  100.664622
     ip_risk_score  0.00    0.770000
   failed_attempts  0.00    3.000000


Como podemos observar, hay varias inconsistencias. 

Por ejmeplo, transaction-hour que deberia tener un rango de 0 a 23, tiene un calor maximo de 578. Lo mismo con cardholder con edades superiores a 2168

In [58]:
df2.count()

transaction_id            4797
timestamp                 4797
amount                    4797
transaction_hour          4722
merchant_category         4797
cardholder_age            4740
cardholder_country        4797
device_type               4797
device_os                 4797
device_trust_score        4746
velocity_last_24h         4734
ip_risk_score             4797
city                      4797
country                   4797
is_foreign_transaction    4748
location_mismatch         4696
auth_method               4797
failed_attempts           4797
is_fraud                  4797
dtype: int64

In [59]:
# Filtramos las filas con rangos imposibles
# Nos quedamos con lo que esté dentro de esos rangos. 
df3 = df2[
    df2["transaction_hour"].between(0, 23) &
    df2["cardholder_age"].between(0, 120) &
    df2["device_trust_score"].between(0, 100)&
    df2["velocity_last_24h"].between(0,10)
]

# df3.count() en Spark devuelve el número total de filas
print(len(df3))

4500


In [60]:
#Calculamos los rangos de valores de cada columna

# 1. Identificamos las columnas numéricas basándonos en df2
# (Equivalente a num_cols = [c for c, t in df2.dtypes if t in ("int", "double")])
num_cols = df2.select_dtypes(include=['number']).columns

# 2. Calculamos min y max sobre df3, transponemos y reseteamos el índice
# (Equivalente al agg + stack de Spark)
df_rangos = df3[num_cols].agg(['min', 'max']).T.reset_index()

# 3. Renombramos las columnas para que coincidan con el alias (column, min, max)
df_rangos.columns = ['column', 'min', 'max']

# 4. Mostramos el resultado
print(df_rangos.to_string(index=False))

            column   min         max
            amount  0.01 8813.402647
  transaction_hour  0.00   23.000000
    cardholder_age 18.00   69.000000
device_trust_score 25.00  100.000000
 velocity_last_24h  0.00    9.000000
     ip_risk_score  0.00    0.740000
   failed_attempts  0.00    3.000000


Ahora que hemos filtrado podemos observar que ya no hay valores atipicos para las variables antes mencionadas. 

In [61]:
# En Spark: df3.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df3.columns])
print("Conteo de nulos por columna:")
print(df3.isnull().sum())

Conteo de nulos por columna:
transaction_id             0
timestamp                  0
amount                     0
transaction_hour           0
merchant_category          0
cardholder_age             0
cardholder_country         0
device_type                0
device_os                  0
device_trust_score         0
velocity_last_24h          0
ip_risk_score              0
city                       0
country                    0
is_foreign_transaction    40
location_mismatch         30
auth_method                0
failed_attempts            0
is_fraud                   0
dtype: int64


In [62]:
#Eliminemos los ultimos nulos que quedan
# En Spark: df4 = df3.dropna()
df3 = df3.dropna()

print(df3.isnull().sum())

transaction_id            0
timestamp                 0
amount                    0
transaction_hour          0
merchant_category         0
cardholder_age            0
cardholder_country        0
device_type               0
device_os                 0
device_trust_score        0
velocity_last_24h         0
ip_risk_score             0
city                      0
country                   0
is_foreign_transaction    0
location_mismatch         0
auth_method               0
failed_attempts           0
is_fraud                  0
dtype: int64


In [63]:
# En Spark: df5 = df4.dropDuplicates()
df3 = df3.drop_duplicates()

print(f"Filas finales (sin duplicados): {len(df3)}")

Filas finales (sin duplicados): 4437


## Subimos a clickhouse los json!

In [64]:
#eliminemos timestamp antes: 
df3 = df3.drop(columns=["timestamp"], errors="ignore")

# 1. Nombre de la tabla

remote_table_name = 'json_limpio_tabla'

# 2. Borramos la anterior para aplicar el cambio de tipo de dato
client.command(f"DROP TABLE IF EXISTS {remote_table_name}")

# 3. Esquema actualizado: transaction_id ahora es String
client.command(f"""
CREATE TABLE {remote_table_name} (
    transaction_id String,
    amount Float64,
    transaction_hour UInt8,
    merchant_category String,
    cardholder_age UInt8,
    cardholder_country String,
    device_type String,
    device_os String,
    device_trust_score Float64,
    velocity_last_24h Float64,
    ip_risk_score Float64,
    city String,
    country String,
    is_foreign_transaction UInt8,
    location_mismatch UInt8,
    auth_method String,
    failed_attempts UInt8,
    is_fraud UInt8
) ENGINE = MergeTree()
ORDER BY transaction_id
""")

# 4. Insertamos df5 (que ya tiene los nulos y duplicados fuera)
client.insert_df(remote_table_name, df3)

print(f"¡Todo listo! Tabla '{remote_table_name}' creada con transaction_id como texto.")

¡Todo listo! Tabla 'json_limpio_tabla' creada con transaction_id como texto.


# Limpieza de Logs (Datos no estructurados)


En esta etapa se trabaja con datos no estructurados provenientes de un archivo de logs del sistema.

A diferencia de los datasets estructurados analizados previamente (CSV y JSON), los logs no presentan un esquema tabular definido, sino que contienen información en formato texto plano.

El objetivo es leer el archivo línea por línea y convertirlo en un DataFrame para luego aplicar procesos de extracción y estructuración de la información relevante.
levante.


## Carga inicial de los archivos Logs

In [92]:
import pandas as pd

# Leemos el archivo de logs como texto plano
# Cada línea del archivo se carga como un string independiente
with open("logs_generados.log", "r") as f:
    lines = f.read().splitlines()

# Convertimos las líneas en un DataFrame de una sola columna
# Esto replica el comportamiento de Spark cuando lee archivos de texto
df_logs = pd.DataFrame(lines, columns=["value"])

# Mostramos las primeras filas para verificar la carga
print(df_logs.head(5))

                                               value    transaction_id  \
0  TransactionHour=10 TX-2024-27677439 User=5885 ...  TX-2024-27677439   
1  TransactionHour=11 TX-2024-43666932 User=9499 ...  TX-2024-43666932   
2  TransactionHour=2 TX-2024-48265545 User=208 Am...  TX-2024-48265545   
3  TransactionHour=18 TX-2024-08606441 User=987 A...  TX-2024-08606441   
4  TransactionHour=22 TX-2024-40141241 User=293 A...  TX-2024-40141241   

   transaction_hour  
0                10  
1                11  
2                 2  
3                18  
4                22  


### Extracción de campos estructurados desde los logs

En esta etapa se comienzan a transformar los datos no estructurados en un formato analizable.

Para ello se utilizan expresiones regulares (regex), que permiten identificar patrones específicos dentro del texto y convertirlos en columnas estructuradas dentro del DataFrame.

In [94]:
import pandas as pd

# Extraemos transaction_id desde la columna "value"
df_logs["transaction_id"] = df_logs["value"].str.extract(r"(TX-\d{4}-\d+)")

# Extraemos transaction_hour desde la columna "value"
df_logs["transaction_hour"] = df_logs["value"].str.extract(r"TransactionHour=(\d+)")

# Convertimos transaction_hour a entero
df_logs["transaction_hour"] = df_logs["transaction_hour"].astype("Int64")

# Mostramos resultados
print(df_logs.head(5))

                                               value    transaction_id  \
0  TransactionHour=10 TX-2024-27677439 User=5885 ...  TX-2024-27677439   
1  TransactionHour=11 TX-2024-43666932 User=9499 ...  TX-2024-43666932   
2  TransactionHour=2 TX-2024-48265545 User=208 Am...  TX-2024-48265545   
3  TransactionHour=18 TX-2024-08606441 User=987 A...  TX-2024-08606441   
4  TransactionHour=22 TX-2024-40141241 User=293 A...  TX-2024-40141241   

   transaction_hour  
0                10  
1                11  
2                 2  
3                18  
4                22  


In [95]:
# =========================
# Información del usuario
# =========================

df_logs["user_id"] = df_logs["value"].str.extract(
    r"User=(\d+)", 
    expand=False
)

df_logs["cardholder_age"] = (
    df_logs["value"]
    .str.extract(r"CardholderAge=(\d+)", expand=False)
    .astype(int)
)

# =========================
# Información de la transacción
# =========================

df_logs["amount"] = (
    df_logs["value"]
    .str.extract(r"Amount=([\d,\.]+)", expand=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df_logs["currency"] = df_logs["value"].str.extract(
    r"Amount=[\d,\.]+\s([A-Z]{3})",
    expand=False
)

df_logs["merchant_category"] = df_logs["value"].str.extract(
    r"MerchantCategory=([A-Za-z]+)",
    expand=False
)

df_logs["method"] = df_logs["value"].str.extract(
    r"Method=([A-Za-z]+)",
    expand=False
)

# Maneja tanto "Status=" como "Statuss="
df_logs["status"] = df_logs["value"].str.extract(
    r"Status[s]?=([A-Za-z]+)",
    expand=False
)

df_logs["is_fraud"] = (
    df_logs["value"]
    .str.extract(r"IsFraud=(true|false)", expand=False)
    .map({"true": True, "false": False})
)


# =========================
# Información de riesgo y comportamiento
# =========================

df_logs["velocity_last_24h"] = (
    df_logs["value"]
    .str.extract(r"VelocityLast24h=([\d\.]+)", expand=False)
    .astype(float)
)

df_logs["device_trust_score"] = (
    df_logs["value"]
    .str.extract(r"DeviceTrustScore=(\d+)", expand=False)
    .astype(float)
)

df_logs["is_foreign_transaction"] = (
    df_logs["value"]
    .str.extract(r"ForeignTransaction=(\d)", expand=False) == "1"
)

df_logs["location_mismatch"] = (
    df_logs["value"]
    .str.extract(r"LocationMismatch=(\d)", expand=False) == "1"
)


# =========================
# Autenticación
# =========================

df_logs["failed_attempts"] = (
    df_logs["value"]
    .str.extract(r"FailedAttempts=(\d+)", expand=False)
    .fillna(0)
    .astype(int)
)


# =========================
# Ubicación e IP
# =========================

df_logs["ip"] = df_logs["value"].str.extract(
    r"IP=([\d\.]+)",
    expand=False
)


# =========================
# Verificación final
# =========================

print(df_logs.head(5))
print(df_logs.info())


                                               value    transaction_id  \
0  TransactionHour=10 TX-2024-27677439 User=5885 ...  TX-2024-27677439   
1  TransactionHour=11 TX-2024-43666932 User=9499 ...  TX-2024-43666932   
2  TransactionHour=2 TX-2024-48265545 User=208 Am...  TX-2024-48265545   
3  TransactionHour=18 TX-2024-08606441 User=987 A...  TX-2024-08606441   
4  TransactionHour=22 TX-2024-40141241 User=293 A...  TX-2024-40141241   

   transaction_hour user_id  cardholder_age     amount currency  \
0                10    5885              70   77480.47      ARS   
1                11    9499              34   40368.68      ARS   
2                 2     208              49  114638.99      ARS   
3                18     987              67   78865.06      ARS   
4                22     293              33  101118.72      ARS   

  merchant_category    method    status  is_fraud  velocity_last_24h  \
0       electronics  Transfer  APPROVED     False                6.7   
1       

Al visualizar las primeras filas del DataFrame estructurado, se observa que la mayoría de las variables fueron correctamente extraídas mediante expresiones regulares.

Las columnas correspondientes a identificadores, montos, variables de riesgo y edad del titular presentan valores coherentes, lo que confirma la correcta detección de los patrones definidos.

### Analizamos como nos quedo el data frame

In [68]:
# Equivalente a df_logs.printSchema()
df_logs.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   value                   3000 non-null   str    
 1   transaction_id          3000 non-null   str    
 2   timestamp               0 non-null      str    
 3   user_id                 3000 non-null   str    
 4   cardholder_age          3000 non-null   int64  
 5   amount                  3000 non-null   float64
 6   currency                3000 non-null   str    
 7   merchant_category       3000 non-null   str    
 8   method                  3000 non-null   str    
 9   status                  3000 non-null   str    
 10  is_fraud                3000 non-null   bool   
 11  velocity_last_24h       3000 non-null   float64
 12  device_trust_score      3000 non-null   float64
 13  is_foreign_transaction  3000 non-null   bool   
 14  location_mismatch       3000 non-null   bool   
 15

### Descripción de variables del dataset de logs

El dataset de logs contiene 3000 registros y 23 columnas, resultantes de la transformación de texto no estructurado en un formato estructurado.

A continuación se describen las principales variables obtenidas:

- value: línea original del log en formato texto plano.
- transaction_id: identificador único de la transacción asociada al evento.
- timestamp: fecha y hora en la que se registró el evento en el log.
- user_id: identificador del usuario asociado a la transacción.
- amount: monto de la transacción.
- currency: moneda en la que se realizó la transacción.
- merchant_category: categoría del comercio.
- method: método utilizado en la transacción.
- status: estado del evento registrado (por ejemplo, aprobado o rechazado).
- is_foreign_transaction: indica si la transacción fue internacional (True/False).
- location_mismatch: indica si existe discrepancia entre la ubicación esperada y la detectada.
- device_trust_score: puntuación de confianza del dispositivo utilizado.
- velocity_last_24h: cantidad de transacciones realizadas en las últimas 24 horas.
- cardholder_age: edad del titular de la tarjeta.
- is_fraud: variable objetivo que indica si la transacción fue fraudulenta (True/False).
- failed_attempts: cantidad de intentos fallidos de autenticación.
- ip: dirección IP desde la cual se realizó la transacción.


In [97]:
#Vemos las columnas que no son vacias

print(
    df_logs[
        [
            "transaction_id",
            "transaction_hour",
            "user_id",
            "cardholder_age",
            "amount",
            "currency",
            "merchant_category",
            "is_fraud",
            "velocity_last_24h",         
            "device_trust_score",
            "is_foreign_transaction",
            "location_mismatch",
            "failed_attempts",           
            "status",
            "ip"
        ]
    ].head(10)
)


     transaction_id  transaction_hour user_id  cardholder_age     amount  \
0  TX-2024-27677439                10    5885              70   77480.47   
1  TX-2024-43666932                11    9499              34   40368.68   
2  TX-2024-48265545                 2     208              49  114638.99   
3  TX-2024-08606441                18     987              67   78865.06   
4  TX-2024-40141241                22     293              33  101118.72   
5  TX-2024-47112582                15    3549              80   62869.75   
6  TX-2024-49317060                 7    7181              49  128447.75   
7  TX-2024-37724473                 8    9403              24    7165.92   
8  TX-2024-45773716                 8     424              50   61770.02   
9  TX-2024-11245804                 4    5627              59   63278.26   

  currency merchant_category  is_fraud  velocity_last_24h  device_trust_score  \
0      ARS       electronics     False                6.7                51.0   
1

In [98]:
#Vemos que tiene la colunma value
print(df_logs.loc[0, "value"])

TransactionHour=10 TX-2024-27677439 User=5885 Amount=77480.47 ARS MerchantCategory=electronics Method=Transfer Statuss=APPROVED ForeignTransaction=1 LocationMismatch=0 DeviceTrustScore=51 VelocityLast24h=6.7 CardholderAge=70 IsFraud=false IP=142.91.243.81


In [99]:
#Como guarda lo mismo que el resto de las columnas pero en una sola, procedemos a borrarla

df_logs = df_logs.drop(columns=["value"])

In [100]:
# Equivalente a df_logs.count()
print(len(df_logs))

3000


In [101]:
#Vemos cuantos nulos tiene cada columna
df_logs.isna().sum()

transaction_id              0
transaction_hour            0
user_id                     0
cardholder_age              0
amount                      0
currency                    0
merchant_category           0
method                      0
status                      0
is_fraud                    0
velocity_last_24h           0
device_trust_score          0
is_foreign_transaction      0
location_mismatch           0
failed_attempts             0
ip                        865
dtype: int64

In [102]:
# Rellenamos los nulos de IP con 0
df_logs["ip"] = df_logs["ip"].fillna(0)

# y verificamos
print(df_logs["ip"].isna().sum())
print(len(df_logs))

0
3000


### Cálculo de la variable ip_risk_score

La column `ip_risk_scoe` no se encuentra explícitamente en los archivos de logs, por lo que se construye de manera derivada utilizando las mismas reglas aplicadas previamente en los archivos JSON.

El objetivo es generarn **indicador numérico de riesgo entre 0 1**, combinando distintas señales de comportamiento y contexto de la transacción.

El cálculo parte de un valor ba de 0.05**, al cual se le suman incrementos según determinadas condiciones:

- Se suma **0.25** si la transacción es internaional (`is_foreign_transactio = True`).
- Se suma **0.25** si existe discrepancia de ubcación (`location_mismath = True`).
- Se suma **0.15** si el monto eselevado (`amunt ≥ 500`).
- Se suma **0.10** si la velocidad transaccional en las últimas 24 hora es alta (`velocity_lat_24h ≥ 10`).
- Se suma **0.10** si el dispositivo presenta bajo nivel deconfianza (`device_trus_score ≤ 40`).

Estas condiciones se impleme"tan mediant" `numpy.where`, lo que permite realizar el cálculo de forma vectorizada sobre todo el DataFrame.

Finalmente, el resultado se norma"iza utiliza"do `.clip(0, 1)` para asegurar que el valor permanezca dentro del rango [0,1], se redondea a dos decimales y se convierte explíctamene a tipo `float`.

De esta manera, se obtiene una variable sintética que resume múltiples factores de riesgo en un único indicador cuantitativo, facilitando su utilización en etapas posteriores de análisis o modelado.


In [103]:
# ============================================
# Cálculo de la variable ip_risk_score
# ============================================

# La columna ip_risk_score no viene en los logs,
# por lo tanto la construimos manualmente combinando distintas señales de riesgo.

import numpy as np

df_logs["ip_risk_score"] = (
    # Valor base de riesgo
    0.05
    
    # +0.25 si la transacción es internacional
    + np.where(df_logs["is_foreign_transaction"] == True, 0.25, 0)
    
    # +0.25 si hay discrepancia de ubicación
    + np.where(df_logs["location_mismatch"] == True, 0.25, 0)
    
    # +0.15 si el monto es mayor o igual a 500
    + np.where(
        df_logs["amount"].notna() & (df_logs["amount"] >= 500),
        0.15,
        0
    )
    
    # +0.10 si la velocidad de transacciones en 24h es mayor o igual a 10
    + np.where(
        df_logs["velocity_last_24h"].notna() & (df_logs["velocity_last_24h"] >= 10),
        0.10,
        0
    )
    
    # +0.10 si el score de confianza del dispositivo es bajo (≤ 40)
    + np.where(
        df_logs["device_trust_score"].notna() & (df_logs["device_trust_score"] <= 40),
        0.10,
        0
    )
)

# ============================================
# Normalización del score
# ============================================

# Aseguramos que el valor quede entre 0 y 1
# (equivalente a greatest(0.0, least(1.0, x)) en Spark)
df_logs["ip_risk_score"] = (
    df_logs["ip_risk_score"]
    .clip(lower=0.0, upper=1.0)  # limita el rango
    .round(2)                   # redondea a 2 decimales
    .astype(float)              # asegura tipo float
)

# Verificamos resultado
print(df_logs[["ip_risk_score"]].head())


   ip_risk_score
0           0.45
1           0.20
2           0.70
3           0.45
4           0.80


In [104]:
list(df_logs.columns)

['transaction_id',
 'transaction_hour',
 'user_id',
 'cardholder_age',
 'amount',
 'currency',
 'merchant_category',
 'method',
 'status',
 'is_fraud',
 'velocity_last_24h',
 'device_trust_score',
 'is_foreign_transaction',
 'location_mismatch',
 'failed_attempts',
 'ip',
 'ip_risk_score']

### Usamos aclaracion obtenida al principio

El log venia con un archivo de aclaraciones que decia lo siguiente: 
"Todos las transacciones que su id empiecen con "TX-2024-4..." tienen que considerarse como fraudes"


In [105]:
#Veamos la cantidad de transacciones fraudes
print(df_logs.groupby("is_fraud").size())

is_fraud
False    2700
True      300
dtype: int64


In [121]:
# Contamos filas donde transaction_id empieza con "TX-2024-4"
cuento_logs = df_logs["transaction_id"].str.startswith("TX-2024-4").sum()

print("Cantidad de filas que cumplen en logs:", cuento_logs)

Cantidad de filas que cumplen en logs: 1000


In [122]:
#Cambiamos todas las transacciones que cumplen la condicion a fraudes

df_logs3 = df_logs.copy()

df_logs3["is_fraud"] = np.where(
    df_logs3["transaction_id"].str.startswith("TX-2024-4"),
    True,
    df_logs3["is_fraud"]
)


In [123]:
#Veamos ahora como quedo
print(df_logs3.groupby("is_fraud").size())

is_fraud
False    1805
True     1195
dtype: int64


In [124]:
#Ahora el dataframe quedo más balanceado

### Validamos rangos de los distintas columnas

En esta etapa se analizan los valores mínimos y máximos de las columnas numéricas del dataset. El objetivo es:
- Detectar valores fuera de rango (por ejemplo, horas mayores a 23).
- Identificar montos negativos o inconsistentes.
- Verificar que las variables cuantitativas respeten los límites esperados.

In [125]:
#Calculamos los rangos de valores de cada columna 
# 1. Identificamos columnas numéricas (Equivalente a num_cols de Spark)
num_cols = df_logs3.select_dtypes(include=['number']).columns

# 2. Calculamos min y max, transponemos (.T) y reseteamos el índice
# Esto hace exactamente lo mismo que el agg + stack de Spark
df_rangos = df_logs3[num_cols].agg(['min', 'max']).T.reset_index() 

# 3. Renombramos las columnas para que queden igual que en tu alias de Spark
df_rangos.columns = ['column', 'min', 'max'] 

# 4. Mostramos el resultado (Equivalente a .show)
print(df_rangos.to_string(index=False))

            column   min       max
  transaction_hour  0.00     23.00
    cardholder_age 18.00     80.00
            amount 83.14 149955.99
 velocity_last_24h  0.00     10.00
device_trust_score  0.00    100.00
   failed_attempts  0.00      0.00
     ip_risk_score  0.15      0.80


Podemos observar que no hay valores extraños en las columnas observadas

### Validación final del DataFrame y eliminación de duplicados

En esta etapa final se realiza una verificación general del DataFrame ya estructurado y limpio.
Para ello: 

- Veremos que no existan valores nulos en alguna columnna.
- Veremos si no existen duplicados exactos. La eliminación de duplicados se realiza considerando filas completamente idénticas en todas sus columnas.


In [126]:
# ============================================
# 1. Información general del DataFrame
# ============================================

print("Dimensión actual del dataset:")
print(df_logs.shape)

print("\nCantidad de nulos por columna:")
print(df_logs.isna().sum())

# ============================================
# 2. Detección de duplicados exactos
# ============================================

duplicados = df_logs.duplicated().sum()
print(f"\nCantidad de filas duplicadas exactas: {duplicados}")

# ============================================
# 3. Eliminación de duplicados
# ============================================

df_logs = df_logs.drop_duplicates()

print("\nDimensión luego de eliminar duplicados:")
print(df_logs.shape)

# ============================================
# 4. Verificación final
# ============================================

print("\nResumen final del DataFrame:")
df_logs.info()


Dimensión actual del dataset:
(3000, 17)

Cantidad de nulos por columna:
transaction_id            0
transaction_hour          0
user_id                   0
cardholder_age            0
amount                    0
currency                  0
merchant_category         0
method                    0
status                    0
is_fraud                  0
velocity_last_24h         0
device_trust_score        0
is_foreign_transaction    0
location_mismatch         0
failed_attempts           0
ip                        0
ip_risk_score             0
dtype: int64

Cantidad de filas duplicadas exactas: 0

Dimensión luego de eliminar duplicados:
(3000, 17)

Resumen final del DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          3000 non-null   str    
 1   transaction_hour        3000 non-null   Int64  
 2   us

Perfecto, el dataset ya quedo listo para subirlo a clickhouse. 

## Importacion a Clickhouse - Logs

In [127]:
# 1. Nombre de la tabla
remote_table_name = 'logs_limpios_tabla'

# 2. Borramos la anterior si existe
client.command(f"DROP TABLE IF EXISTS {remote_table_name}")

# 3. Creamos la tabla con el esquema correcto
client.command(f"""
CREATE TABLE {remote_table_name} (
    transaction_id String,
    transaction_hour UInt8,
    user_id String,
    cardholder_age UInt8,
    amount Float64,
    currency String,
    merchant_category String,
    is_fraud UInt8,
    velocity_last_24h Float64,
    device_trust_score Float64,
    is_foreign_transaction UInt8,
    location_mismatch UInt8,
    failed_attempts UInt8,
    ip String,
    method String,
    status String,
    ip_risk_score Float64
) ENGINE = MergeTree()
ORDER BY transaction_id
""")


In [128]:
#Ajustamos tipos antes de insertar
#ClickHouse no acepta boolean directamente → lo pasamos a UInt8.

df_click = df_logs.copy()

df_click["is_fraud"] = df_click["is_fraud"].astype(int)
df_click["is_foreign_transaction"] = df_click["is_foreign_transaction"].astype(int)
df_click["location_mismatch"] = df_click["location_mismatch"].astype(int)


In [129]:
client.insert_df(remote_table_name, df_click)

print(f"¡Todo listo! Tabla '{remote_table_name}' creada y datos insertados")

¡Todo listo! Tabla 'logs_limpios_tabla' creada y datos insertados


In [130]:

# Definimos la lógica final: IDs como String y Categorías en Minúsculas
query_vista_final = """
CREATE OR REPLACE VIEW vista_fraude_unificada AS
SELECT 
    transaction_id,
    argMax(amount, fuente) as amount,
    argMax(transaction_hour, fuente) as transaction_hour,
    argMax(merchant_category, fuente) as merchant_category,
    argMax(foreign_transaction, fuente) as foreign_transaction,
    argMax(location_mismatch, fuente) as location_mismatch,
    argMax(device_trust_score, fuente) as device_trust_score,
    argMax(velocity_last_24h, fuente) as velocity_last_24h,
    argMax(cardholder_age, fuente) as cardholder_age,
    argMax(is_fraud, fuente) as is_fraud
FROM (
    -- 1. CSV
    SELECT 
        toString(transaction_id) as transaction_id, 
        toFloat64(amount) as amount, 
        toInt8(transaction_hour) as transaction_hour, 
        lower(merchant_category) as merchant_category, --Estandarización
        toInt8(foreign_transaction) as foreign_transaction, 
        toInt8(location_mismatch) as location_mismatch, 
        toFloat64(device_trust_score) as device_trust_score, 
        toFloat64(velocity_last_24h) as velocity_last_24h, 
        toInt16(cardholder_age) as cardholder_age, 
        toInt8(is_fraud) as is_fraud, 
        1 as fuente 
    FROM mi_tabla_limpia

    UNION ALL

    -- 2. JSON
    SELECT 
        transaction_id, 
        toFloat64(amount), 
        toInt8(transaction_hour), 
        lower(merchant_category), -- Estandarización
        toInt8(is_foreign_transaction) as foreign_transaction, 
        toInt8(location_mismatch), 
        toFloat64(device_trust_score), 
        toFloat64(velocity_last_24h), 
        toInt16(cardholder_age), 
        toInt8(is_fraud), 
        2 as fuente 
    FROM json_limpio_tabla

    UNION ALL

    -- 3. LOGS
    SELECT 
        transaction_id, 
        toFloat64(amount), 
        toInt8(transaction_hour), 
        lower(merchant_category), -- Estandarización
        toInt8(is_foreign_transaction) as foreign_transaction, 
        toInt8(location_mismatch), 
        toFloat64(device_trust_score), 
        toFloat64(velocity_last_24h), 
        toInt16(cardholder_age), 
        toInt8(is_fraud), 
        3 as fuente 
    FROM logs_limpios_tabla
)
GROUP BY transaction_id
"""

client.command(query_vista_final)
print("Vista final optimizada: IDs unificados, categorías en minúsculas y tipos de datos alineados.")

✅ Vista final optimizada: IDs unificados, categorías en minúsculas y tipos de datos alineados.


In [131]:
# Consulta para encontrar IDs repetidos
query_duplicados = """
SELECT 
    transaction_id, 
    count() as repeticiones
FROM vista_fraude_unificada
GROUP BY transaction_id
HAVING repeticiones > 1
ORDER BY repeticiones DESC
LIMIT 10
"""

df_duplicados = client.query_df(query_duplicados)

if df_duplicados.empty:
    print("¡Perfecto! No hay IDs duplicados.")
else:
    print(f" Atención: Se encontraron {len(df_duplicados)} IDs duplicados.")
    print(df_duplicados.head())

✅ ¡Perfecto! No hay IDs duplicados.


In [132]:
df_duplicados.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame
